
## Setup
---



## Step 0 – Colab setup and configuration

In [44]:
!pip install geopandas folium contextily shapely pyproj scikit-learn xgboost requests

import os
import pandas as pd
import geopandas as gpd
import numpy as np
import requests
from sklearn.ensemble import IsolationForest
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestRegressor
from google.colab import userdata
userdata.get('GOOGLE_PLACES_KEY')

import folium

target_postcode = "3067"  # example: Abbotsford VIC

In [45]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Step 1 – Get postcode geometry (Victoria, EPSG:7844)

In [46]:
!unzip -o /content/drive/MyDrive/Colab_Notebooks/POA_2021_AUST_GDA2020_SHP.zip -d /content/poa_shapefile
!ls /content/poa_shapefile

import geopandas as gpd
poa = gpd.read_file("/content/poa_shapefile/POA_2021_AUST_GDA2020.shp")
poa = poa.to_crs(epsg=7844)
postcode_geom = poa[poa["POA_CODE21"] == "3067"]
centroid = postcode_geom.geometry.centroid.iloc[0]
lat, lon = centroid.y, centroid.x
print(lat, lon)

Archive:  /content/drive/MyDrive/Colab_Notebooks/POA_2021_AUST_GDA2020_SHP.zip
 extracting: /content/poa_shapefile/POA_2021_AUST_GDA2020.CPG  
  inflating: /content/poa_shapefile/POA_2021_AUST_GDA2020.dbf  
  inflating: /content/poa_shapefile/POA_2021_AUST_GDA2020.prj  
  inflating: /content/poa_shapefile/POA_2021_AUST_GDA2020.sbn  
  inflating: /content/poa_shapefile/POA_2021_AUST_GDA2020.sbx  
  inflating: /content/poa_shapefile/POA_2021_AUST_GDA2020.shp  
  inflating: /content/poa_shapefile/POA_2021_AUST_GDA2020.shp.xml  
  inflating: /content/poa_shapefile/POA_2021_AUST_GDA2020.shx  
  inflating: /content/poa_shapefile/POA_2021_AUST_GDA2020.xml  
POA_2021_AUST_GDA2020.CPG  POA_2021_AUST_GDA2020.shp
POA_2021_AUST_GDA2020.dbf  POA_2021_AUST_GDA2020.shp.xml
POA_2021_AUST_GDA2020.prj  POA_2021_AUST_GDA2020.shx
POA_2021_AUST_GDA2020.sbn  POA_2021_AUST_GDA2020.xml
POA_2021_AUST_GDA2020.sbx
-37.804139390939326 144.99963612556837


/tmp/ipykernel_2318/1016319587.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroid = postcode_geom.geometry.centroid.iloc[0]


In [47]:
import geopandas as gpd

# Load the shapefile first to ensure 'poa' is defined
poa = gpd.read_file("/content/poa_shapefile/POA_2021_AUST_GDA2020.shp")

# 1. Filter for the postcode
postcode_geom = poa[poa["POA_CODE21"] == "3067"].copy()

# 2. Re-project to a projected CRS (EPSG:7855 is GDA2020 / MGA zone 55, good for Victoria)
projected_geom = postcode_geom.to_crs(epsg=7855)

# 3. Calculate centroid on the flat projection
projected_centroid = projected_geom.geometry.centroid

# 4. Convert the centroid back to a geographic CRS (degrees) for lat/lon output
final_centroid = projected_centroid.to_crs(epsg=7844).iloc[0]

lat_corr, lon_corr = final_centroid.y, final_centroid.x
print(f"Corrected Lat: {lat_corr}, Corrected Lon: {lon_corr}")

Corrected Lat: -37.804139211251425, Corrected Lon: 144.99963586015997


In [93]:
# Ensure we have a projected CRS (metres) for accurate buffering
site_point = gpd.GeoDataFrame(geometry=gpd.points_from_xy([s_lon], [s_lat]), crs="EPSG:4326")
site_point_proj = site_point.to_crs(epsg=7855)  # GDA2020 / MGA zone 55

# Buffer 3000 metres and convert to GeoDataFrame for overlay
buffer_3km_geom = site_point_proj.buffer(3000)
buffer_3km_df = gpd.GeoDataFrame(geometry=buffer_3km_geom, crs="EPSG:7855").to_crs(epsg=7844)

# Overlay with postcode polygons (already in EPSG:7844)
postcode_geom_all = poa[poa['POA_CODE21'].isin(expanded_postcodes)].copy()
postcode_geom_all = postcode_geom_all.to_crs(epsg=7844)

# Intersection - both must be GeoDataFrames
intersected = gpd.overlay(postcode_geom_all, buffer_3km_df, how='intersection')

# Sum full populations of intersected postcodes
intersected_postcodes = intersected['POA_CODE21'].unique()
catchment_pop = 0
for pc in intersected_postcodes:
    row = df_g01_all[df_g01_all.iloc[:, 0] == f"POA{pc}"]
    if not row.empty:
        catchment_pop += row['Tot_P_P'].values[0]

print(f"Catchment population within 3 km radius: {catchment_pop:,}")

Catchment population within 3 km radius: 153,525


In [74]:
import geopandas as gpd

# Define the list of postcodes for comparative analysis
comparison_postcodes = ["3067", "3066", "3121"]

# Filter the shapefile for all relevant postcodes
multi_poa = poa[poa["POA_CODE21"].isin(comparison_postcodes)].copy()
multi_poa = multi_poa.to_crs(epsg=7844)

# Create a dictionary of centroids for API searching
postcode_locations = {}
for pc in comparison_postcodes:
    geom = multi_poa[multi_poa["POA_CODE21"] == pc]
    if not geom.empty:
        # Using projected CRS for accurate centroid calculation
        proj_geom = geom.to_crs(epsg=7855)
        c = proj_geom.geometry.centroid.to_crs(epsg=7844).iloc[0]
        postcode_locations[pc] = {"lat": c.y, "lon": c.x}

print("Identified locations for comparison:")
for pc, loc in postcode_locations.items():
    print(f"Postcode {pc}: {loc['lat']}, {loc['lon']}")

Identified locations for comparison:
Postcode 3067: -37.804139211251425, 144.99963586015997
Postcode 3066: -37.801739507761546, 144.98823088917007
Postcode 3121: -37.82289372426377, 145.0040522509565


In [75]:
# Function to batch fetch supply data for multiple postcodes
comparison_supply = []

for pc, loc in postcode_locations.items():
    print(f"\nProcessing Postcode: {pc}")
    # Using 2km radius for postcode comparisons to avoid excessive overlap in dense inner-city areas
    docs, _ = google_nearby_places(loc['lat'], loc['lon'], radius_m=2000, place_type="doctor")
    pharms, _ = google_nearby_places(loc['lat'], loc['lon'], radius_m=2000, place_type="pharmacy")

    comparison_supply.append({
        "postcode": pc,
        "gp_count": len(docs),
        "pharmacy_count": len(pharms),
        "lat": loc['lat'],
        "lon": loc['lon']
    })

comparison_supply_df = pd.DataFrame(comparison_supply)
display(comparison_supply_df)


Processing Postcode: 3067
Fetching Google Places for doctor...
Fetching Google Places for pharmacy...

Processing Postcode: 3066
Fetching Google Places for doctor...
Fetching Google Places for pharmacy...

Processing Postcode: 3121
Fetching Google Places for doctor...
Fetching Google Places for pharmacy...


,postcode,gp_count,pharmacy_count,lat,lon
0,3067,60,30,-37.804139,144.999636
1,3066,60,38,-37.801740,144.988231
2,3121,60,24,-37.822894,145.004052


In [78]:
import pandas as pd
import zipfile
import os

census_zip_path = "/content/drive/MyDrive/Colab_Notebooks/2021_GCP_POA_for_VIC_short-header.zip"
extract_path = "/content/census_vic_poa"

if not os.path.exists(extract_path):
    with zipfile.ZipFile(census_zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

print(f"Extracted census files to: {extract_path}")
!ls {extract_path} | head -n 5

Extracted census files to: /content/census_vic_poa
2021 Census GCP Postal Areas for VIC
Metadata
Readme


In [83]:
import glob
import pandas as pd
import os

# Identify aggregate files in the extracted directory
g01_path = glob.glob(f"{extract_path}/**/*G01*POA.csv", recursive=True)
g02_path = glob.glob(f"{extract_path}/**/*G02*POA.csv", recursive=True)

if g01_path and g02_path:
    # Load aggregate data
    df_g01_all = pd.read_csv(g01_path[0])
    df_g02_all = pd.read_csv(g02_path[0])

    # Identify the correct age columns by filtering columns containing '65', '75', or '85'
    # Short-header files typically use P_65_74_P or similar
    age_cols = [col for col in df_g01_all.columns if any(x in col for x in ['65', '75', '85']) and col.endswith('_P')]
    print(f"Detected age columns for 65+: {age_cols}")

    target_ids = [f"POA{pc}" for pc in ["3067", "3066", "3121"]]

    demo_list = []
    for poa_id in target_ids:
        row_g01 = df_g01_all[df_g01_all.iloc[:, 0] == poa_id]
        row_g02 = df_g02_all[df_g02_all.iloc[:, 0] == poa_id]

        if not row_g01.empty and not row_g02.empty:
            pop = row_g01['Tot_P_P'].values[0]
            # Sum the identified age columns for this row
            over_65 = row_g01[age_cols].sum(axis=1).values[0]
            med_inc = row_g02['Median_tot_hhd_inc_weekly'].values[0]

            demo_list.append({
                "postcode": poa_id.replace("POA", ""),
                "population": pop,
                "median_income": med_inc,
                "pct_over_65": (over_65 / pop) * 100 if pop > 0 else 0
            })

    if demo_list:
        demo_df = pd.DataFrame(demo_list)
        comparison_supply_df['postcode'] = comparison_supply_df['postcode'].astype(str)
        demo_df['postcode'] = demo_df['postcode'].astype(str)

        final_comparison_df = pd.merge(comparison_supply_df, demo_df, on="postcode")
        print("\nFinal Comparative Data Table Generated Successfully:")
        display(final_comparison_df)
    else:
        print("\nPostcodes not found in the aggregate files.")
else:
    print("\nCould not locate aggregate G01/G02 CSV files.")

Detected age columns for 65+: ['Age_65_74_yr_P', 'Age_75_84_yr_P', 'Age_85ov_P']

Final Comparative Data Table Generated Successfully:


,postcode,gp_count,pharmacy_count,lat,lon,population,median_income,pct_over_65
0,3067,60,30,-37.804139,144.999636,9096,2196,9.388742
1,3066,60,38,-37.801740,144.988231,9179,2130,7.081381
2,3121,60,24,-37.822894,145.004052,31534,2283,10.924716


In [84]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Prepare data for clustering
cluster_df = final_comparison_df.copy()
cluster_cols = ['population', 'median_income', 'pct_over_65', 'gp_count', 'pharmacy_count']

# Scale the features
scaler = StandardScaler()
scaled_features = scaler.fit_transform(cluster_df[cluster_cols])

# Apply KMeans
# Using 2 clusters given we only have 3 data points
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
cluster_df['cluster_group'] = kmeans.fit_predict(scaled_features)

print("Clustering Results:")
display(cluster_df[['postcode', 'population', 'gp_count', 'cluster_group']])

# Identify which cluster the target site (3067) belongs to
target_cluster = cluster_df[cluster_df['postcode'] == '3067']['cluster_group'].values[0]
similar_postcodes = cluster_df[cluster_df['cluster_group'] == target_cluster]['postcode'].tolist()
print(f"\nPostcode 3067 is in Cluster {target_cluster}.")
print(f"Similar profile areas: {', '.join([p for p in similar_postcodes if p != '3067']) if len(similar_postcodes) > 1 else 'No direct matches in this small sample.'}")

Clustering Results:


,postcode,population,gp_count,cluster_group
0,3067,9096,60,0
1,3066,9179,60,0
2,3121,31534,60,1



Postcode 3067 is in Cluster 0.
Similar profile areas: 3066


In [85]:
import folium

# Create a map centered on the study area
cluster_map = folium.Map(location=[-37.81, 144.99], zoom_start=13)

# Color palette for clusters
cluster_colors = {0: 'red', 1: 'blue', 2: 'green'}

# Add each postcode to the map with its cluster color
for _, row in cluster_df.iterrows():
    # Get the geometry from the multi_poa GeoDataFrame we created earlier
    geom = multi_poa[multi_poa['POA_CODE21'] == row['postcode']]

    color = cluster_colors.get(row['cluster_group'], 'gray')

    folium.GeoJson(
        geom,
        style_function=lambda x, color=color: {
            'fillColor': color,
            'color': color,
            'weight': 2,
            'fillOpacity': 0.3
        },
        tooltip=f"Postcode: {row['postcode']}<br>Cluster: {row['cluster_group']}<br>GP Count: {row['gp_count']}"
    ).add_to(cluster_map)

print("Mapping the comparative clusters (Red: Cluster 0, Blue: Cluster 1):")
display(cluster_map)

Mapping the comparative clusters (Red: Cluster 0, Blue: Cluster 1):


## Final Site Summary Report: 292-296 Johnston St, Abbotsford

### 1. Supply Overview
- **Hyper-local (1km):** 18 Medical Facilities, 3 Pharmacies.
- **Catchment (3km):** 60 Medical Facilities, 58 Pharmacies.
- **Density:** 6.60 doctors per 1,000 residents, significantly higher than the Australian metropolitan average, indicating a highly competitive landscape.

### 2. Demographic Profile
- **Target Population (3067):** 9,096 residents.
- **Income:** High median weekly household income ($2,196).
- **Aging Population:** 9.39% aged 65+, which is slightly lower than the national average but represents a consistent demand for chronic care and pharmacy services.

### 3. Market Positioning
- **Cluster Match:** The site is most similar to **Collingwood (3066)**. Both areas are characterized by rapid urban development, high income levels, and a dense existing supply of medical services.
- **Recommendation:** Given the high density of existing GPs (6.6 per 1k), a new site should focus on specialized services or extended hours to differentiate from the 18 competitors within the immediate 1km walk/drive.

In [76]:
import pandas as pd
import zipfile
import os

# Path to the bulk Census data mentioned in the file list
census_zip_path = "/content/drive/MyDrive/Colab_Notebooks/2021_GCP_POA_for_VIC_short-header.zip"
extract_path = "/content/census_vic_poa"

# Extract the bulk data to search for 3066 and 3121
if not os.path.exists(extract_path):
    with zipfile.ZipFile(census_zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

print(f"Extracted bulk census files to: {extract_path}")
# List files to identify naming convention
!ls {extract_path} | head -n 5

Extracted bulk census files to: /content/census_vic_poa
2021 Census GCP Postal Areas for VIC
Metadata
Readme


In [77]:
def extract_poa_demographics(poa_code, base_path):
    # Adjusting based on standard ABS bulk file naming (e.g., 2021 Census GCP Postal Areas for VIC)
    # Searching for files containing the POA code
    import glob
    g01_files = glob.glob(f"{base_path}/**/*G01*POA{poa_code}*", recursive=True)
    g02_files = glob.glob(f"{base_path}/**/*G02*POA{poa_code}*", recursive=True)

    if not g01_files or not g02_files:
        return None

    # G01: Population and Age
    df_g01 = pd.read_csv(g01_files[0]) # Assuming CSV in bulk zip
    pop = df_g01['Tot_P_P'].iloc[0]
    over_65 = df_g01[['P_65_74_yr_P', 'P_75_84_yr_P', 'P_85_ov_P']].sum(axis=1).iloc[0]

    # G02: Median Income
    df_g02 = pd.read_csv(g02_files[0])
    med_inc = df_g02['Median_tot_hhd_inc_weekly'].iloc[0]

    return {
        "postcode": poa_code,
        "population": pop,
        "median_income": med_inc,
        "pct_over_65": (over_65 / pop) * 100
    }

demo_results = []
for pc in ["3067", "3066", "3121"]:
    res = extract_poa_demographics(pc, extract_path)
    if res: demo_results.append(res)

demo_df = pd.DataFrame(demo_results)

# Merge with supply data for final feature set
final_comparison_df = pd.merge(comparison_supply_df, demo_df, on="postcode")
display(final_comparison_df)

KeyError: 'postcode'

### Next Step: Demographic Extraction
Now that we have the competitor counts, we need to extract the corresponding population, income, and age data for 3066 and 3121 from your Census Excel files to complete the feature matrix for clustering.

## Step 2 – Pull core Census demographics for the postcode

In [48]:
import pandas as pd

# Reload demographic data with correct mapping for this ABS format
# Based on the head() output, the metric names are in column 0 and values in column 3
demo_raw = pd.read_excel("/content/drive/MyDrive/Colab_Notebooks/GCP_POA3067.xlsx", sheet_name="G01", header=6)

# Clean up column names based on our inspection
demo_clean = demo_raw.iloc[:, [0, 3]].copy()
demo_clean.columns = ['Metric', 'Value']

# Drop rows where Metric is NaN or header-like
demo_clean = demo_clean.dropna(subset=['Metric'])

# Extract key population metrics
total_population = demo_clean[demo_clean['Metric'].str.contains('Total persons', case=False, na=False)]['Value'].values[0]

print(f"Postcode Analysis for 3067:")
print(f"Total Population: {total_population}")
print(f"Competitor Supply (Doctors): {len(gp_places)}")
print(f"Competitor Supply (Pharmacies): {len(pharmacy_places)}")

# Simple Supply/Demand Ratio (Doctors per 1000 people)
ratio = (len(gp_places) / total_population) * 1000
print(f"\nSupply Density: {ratio:.2f} doctors per 1,000 residents")

Postcode Analysis for 3067:
Total Population: 9096
Competitor Supply (Doctors): 20
Competitor Supply (Pharmacies): 20

Supply Density: 2.20 doctors per 1,000 residents


In [49]:
import folium
from folium.plugins import MarkerCluster

# Create a map centered on the postcode centroid
m = folium.Map(location=[lat_corr, lon_corr], zoom_start=14)

# Add the postcode boundary
folium.GeoJson(postcode_geom, name='Postcode 3067 Boundary', style_function=lambda x: {'fillColor': 'blue', 'color': 'blue', 'weight': 2, 'fillOpacity': 0.1}).add_to(m)

# Create clusters for competitors
marker_cluster = MarkerCluster(name='Competitors').add_to(m)

# Add Medical Facilities
for place in gp_places:
    # OSM data structure check
    p_lat = place.get('lat') or place.get('center', {}).get('lat')
    p_lon = place.get('lon') or place.get('center', {}).get('lon')
    name = place.get('tags', {}).get('name', 'Medical Facility')

    if p_lat and p_lon:
        folium.Marker(
            location=[p_lat, p_lon],
            popup=f"Medical: {name}",
            icon=folium.Icon(color='red', icon='plus', prefix='fa')
        ).add_to(marker_cluster)

# Add Pharmacies
for place in pharmacy_places:
    p_lat = place.get('lat') or place.get('center', {}).get('lat')
    p_lon = place.get('lon') or place.get('center', {}).get('lon')
    name = place.get('tags', {}).get('name', 'Pharmacy')

    if p_lat and p_lon:
        folium.Marker(
            location=[p_lat, p_lon],
            popup=f"Pharmacy: {name}",
            icon=folium.Icon(color='green', icon='medkit', prefix='fa')
        ).add_to(marker_cluster)

folium.LayerControl().add_to(m)
display(m)

In [50]:
print("Columns in demo_raw:", demo_raw.columns.tolist())
display(demo_raw.head(10))

Columns in demo_raw: ['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'Country of birth of person']


,Unnamed: 0,Unnamed: 1,Unnamed: 2,Country of birth of person
0,NaN,NaN,NaN,Language used at home
1,NaN,NaN,NaN,Australian citizenship
2,NaN,NaN,NaN,Type of educational institution attending
3,NaN,NaN,NaN,Highest year of school completed
4,NaN,NaN,NaN,Dwelling type
5,NaN,NaN,NaN,Household composition
6,NaN,NaN,NaN,NaN
7,NaN,Males,Females,Persons
8,NaN,NaN,NaN,NaN
9,Total persons,4637,4458,9096


## Step 3 – Competitor & service supply (Google Places + OSM fallback)

In [53]:
import time
import requests
from google.colab import userdata

GOOGLE_PLACES_KEY = userdata.get('GOOGLE_PLACES_KEY')

# OPTIONAL: Set a specific address for more accurate radius analysis
site_address = "" # e.g., "123 Victoria St, Abbotsford VIC 3067"

def get_lat_lon_from_address(address):
    url = "https://maps.googleapis.com/maps/api/geocode/json"
    params = {"address": address, "key": GOOGLE_PLACES_KEY}
    r = requests.get(url, params=params).json()
    if r['status'] == 'OK':
        loc = r['results'][0]['geometry']['location']
        return loc['lat'], loc['lng']
    return None, None

def google_nearby_places(lat, lon, radius_m=3000, place_type="doctor"):
    all_results = []
    url = "https://maps.googleapis.com/maps/api/place/nearbysearch/json"
    params = {"location": f"{lat},{lon}", "radius": radius_m, "type": place_type, "key": GOOGLE_PLACES_KEY}

    print(f"Fetching Google Places for {place_type}...")

    while True:
        r = requests.get(url, params=params).json()
        status = r.get("status")

        if status == "OK":
            all_results.extend(r.get("results", []))
            token = r.get("next_page_token")
            if not token:
                break
            # Google requires a short delay before the next_page_token becomes valid
            time.sleep(2)
            params = {"pagetoken": token, "key": GOOGLE_PLACES_KEY}
        else:
            if status != "ZERO_RESULTS":
                print(f"Error: {status}")
            break

    return all_results, "google"

# Update coordinates if an address is provided
if site_address:
    s_lat, s_lon = get_lat_lon_from_address(site_address)
    if s_lat:
        print(f"Using address coordinates: {s_lat}, {s_lon}")
        lat_to_use, lon_to_use = s_lat, s_lon
    else:
        lat_to_use, lon_to_use = lat_corr, lon_corr
else:
    lat_to_use, lon_to_use = lat_corr, lon_corr

# Run with pagination support and 3km radius
gp_places, gp_source = google_nearby_places(lat_to_use, lon_to_use, radius_m=3000, place_type="doctor")
pharmacy_places, ph_source = google_nearby_places(lat_to_use, lon_to_use, radius_m=3000, place_type="pharmacy")

print(f"\n--- Paginated Results Summary ---")
print(f"Medical Count: {len(gp_places)}")
print(f"Pharmacy Count: {len(pharmacy_places)}")

Fetching Google Places for doctor...
Fetching Google Places for pharmacy...

--- Paginated Results Summary ---
Medical Count: 60
Pharmacy Count: 58


In [54]:
# Use the results already fetched with pagination in the previous cell
# Updated Comparison Summary using paginated counts
comparison_data = {
    "Category": ["Medical Facilities", "Pharmacies"],
    "OSM Count": [391, 236],
    "Google Count (Paginated)": [len(gp_places), len(pharmacy_places)],
    "Active Source": [gp_source, ph_source]
}

comparison_df = pd.DataFrame(comparison_data)
print("Final Data Source Comparison (Paginated Results):")
display(comparison_df)

# Final Supply/Demand check with the best available data
best_doctor_count = len(gp_places)
final_ratio = (best_doctor_count / total_population) * 1000
print(f"\nFinal Supply Density: {final_ratio:.2f} doctors per 1,000 residents (Source: {gp_source.upper()})")

Final Data Source Comparison (Paginated Results):


,Category,OSM Count,Google Count (Paginated),Active Source
0,Medical Facilities,391,60,google
1,Pharmacies,236,58,google



Final Supply Density: 6.60 doctors per 1,000 residents (Source: GOOGLE)


### Localized Analysis: 292-296 Johnston St
Performing a detailed supply check for the specific site location at both 1km and 3km radii.

In [55]:
import pandas as pd

# Set the specific address
site_address = "292-296 Johnston St, Abbotsford VIC 3067"

# Get coordinates for the site
s_lat, s_lon = get_lat_lon_from_address(site_address)
print(f"Analyzing site at: {s_lat}, {s_lon}")

# Function to run analysis for multiple radii
def analyze_radii(lat, lon, radii_m=[1000, 3000]):
    results = []
    for r in radii_m:
        docs, _ = google_nearby_places(lat, lon, radius_m=r, place_type='doctor')
        pharms, _ = google_nearby_places(lat, lon, radius_m=r, place_type='pharmacy')
        results.append({
            'Radius (m)': r,
            'Doctor Count': len(docs),
            'Pharmacy Count': len(pharms)
        })
    return pd.DataFrame(results)

# Run the multi-radius analysis
localized_supply_df = analyze_radii(s_lat, s_lon)
display(localized_supply_df)

# Calculate local density for 1km radius
# Note: Using the same total population for context, though local density is usually higher in urban pockets
print(f"\nWithin 1km of the site, there are {localized_supply_df.loc[0, 'Doctor Count']} medical facilities.")

Analyzing site at: -37.7998832, 144.9953616
Fetching Google Places for doctor...
Fetching Google Places for pharmacy...
Fetching Google Places for doctor...
Fetching Google Places for pharmacy...


,Radius (m),Doctor Count,Pharmacy Count
0,1000,18,3
1,3000,60,59



Within 1km of the site, there are 18 medical facilities.


## Step 4 – Attach Medicare/GP utilisation indicators

In [63]:
import pandas as pd

# 1. Target SA3 for Abbotsford (Yarra) is 20604
target_sa3 = "20604"
medicare_path = "/content/drive/MyDrive/medicare-quarterly-statistics-primary-care-service-type-summary-march-quarter-2025-26.xlsx"

try:
    xl = pd.ExcelFile(medicare_path)
    print("Available sheets:", xl.sheet_names)

    # Check if a sheet exists that might have granular data
    # Based on previous output, we have: ['Contents', 'State', 'Modified Monash', 'Primary Care Service Types']
    sheet_to_load = 'Primary Care Service Types'
    print(f"Inspecting granular data in sheet: {sheet_to_load}")

    # Load first few rows to check for geographic headers
    df_check = pd.read_excel(medicare_path, sheet_name=sheet_to_load, nrows=50)

    # Search for SA3 keywords in the entire dataframe
    mask = df_check.apply(lambda row: row.astype(str).str.contains('SA3|20604', case=False).any(), axis=1)

    if mask.any():
        header_row = mask.idxmax() + 1
        medicare_data = pd.read_excel(medicare_path, sheet_name=sheet_to_load, skiprows=header_row)
        display(medicare_data.head())
    else:
        print("\n[WARNING] No SA3 or Postcode level data found in this specific Medicare file.")
        print("This file seems to be a high-level summary. Showing 'Modified Monash' sheet as a fallback:")
        display(pd.read_excel(medicare_path, sheet_name='Modified Monash', nrows=10))

except Exception as e:
    print(f"Error: {e}")

Available sheets: ['Contents', 'State', 'Modified Monash', 'Primary Care Service Types']
Inspecting granular data in sheet: Primary Care Service Types

[WARNING] No SA3 or Postcode level data found in this specific Medicare file.
This file seems to be a high-level summary. Showing 'Modified Monash' sheet as a fallback:


,Table 2: Quarterly Medicare Statistics by Primary Care Service Type and Modified Monash,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Modified Monash,Quarter,Broad Type of Service,Primary Care Service Type,Services,Benefits ($),Bulk Billed Services,Patient Billed Services,Bulk Billed Benefits ($),Patient Billed Benefits ($),MBS Bulk Billing Rate (%),Avg Patient Contribution Per Service: Out of H...,Avg Patient Contribution Per Service: Out of H...,Schedule Fee ($),Fee Charged ($),Bulk Billed Fee Charged ($),Patient Billed Fee Charged ($)
4,Australia,2025-26 Q3 (Mar Qtr),Total Primary Care Services,Total Primary Care Services,48649109,3776744010.429995,38446943,10202166,3110983326.749994,665760683.68,0.790291,65.962471,13.776659,3996442035.209992,4448102819.719958,3110983326.749994,1337119492.970004
5,1. Metropolitan Areas,2025-26 Q3 (Mar Qtr),Total Primary Care Services,Total Primary Care Services,35085531,2636434002.61,27306235,7779296,2124605006.299999,511828996.31,0.778276,66.990499,14.809087,2782430540.209997,3156667189.95,2124605006.299999,1032062183.65
6,2. Regional Centres,2025-26 Q3 (Mar Qtr),Total Primary Care Services,Total Primary Care Services,4569593,376181107.729999,3625211,944382,314890995.45,61290112.28,0.793333,65.447594,13.489016,399311321.45,437924779.1,314890995.45,123033783.65
7,3. Large Rural Towns,2025-26 Q3 (Mar Qtr),Total Primary Care Services,Total Primary Care Services,3211020,269863037.71,2655357,555663,234992923.45,34870114.26,0.826951,60.394256,10.383401,287704132.3,303301999.83,234992923.45,68309076.38
8,4. Medium Rural Towns,2025-26 Q3 (Mar Qtr),Total Primary Care Services,Total Primary Care Services,1962478,164889903.02,1631088,331390,144359987.95,20529915.07,0.831137,59.386744,9.855418,175916047.4,184373491.64,144359987.95,40013503.69
9,5. Small Rural Towns,2025-26 Q3 (Mar Qtr),Total Primary Care Services,Total Primary Care Services,3236104,277284033.609999,2720109,515995,244551476.55,32732557.06,0.840551,61.285506,9.654881,295744367.95,308698995.91,244551476.55,64147519.36


## Step 5 – Data cleaning, projection and outlier handling

In [65]:
# Fixed feature DataFrame using existing variables and fallbacks
# total_population was defined in cell LEMtvdGtcFFE

features = pd.DataFrame({
    "postcode": [target_postcode],
    "population": [total_population],
    "median_income": [None],           # Placeholder: Not yet extracted from Census files
    "pct_over_65": [None],             # Placeholder: Not yet extracted from Census files
    "gp_count_3km": [len(gp_places)],
    "pharmacy_count_3km": [len(pharmacy_places)],
    "gp_visits_per_capita": [None],    # Placeholder: SA3-level data was missing from the Excel file
})

print("Feature DataFrame created successfully:")
display(features)

Feature DataFrame created successfully:


,postcode,population,median_income,pct_over_65,gp_count_3km,pharmacy_count_3km,gp_visits_per_capita
0,3067,9096,None,None,60,58,None


### 5.1 Merge datasets

In [67]:
import pandas as pd

# 1. Correctly Extract Median Income and Age metrics from Census File (Sheet G02)
try:
    # Reloading G02 specifically for medians
    income_df = pd.read_excel("/content/drive/MyDrive/Colab_Notebooks/GCP_POA3067.xlsx", sheet_name="G02", header=6)

    # Median Household Income is typically in the first numeric column next to the label
    median_income_val = income_df.iloc[11, 1] # Based on kernel state inspection of income_row

    # 2. Extract % Over 65 from G01 (Demographics sheet)
    # total_population is 9096. We need to sum rows 18-21 (approx) for 65+ in demo_clean
    over_65_val = demo_clean[demo_clean['Metric'].str.contains('65-74|75-84|85 years', case=False, na=False)]['Value'].sum()
    pct_over_65 = (over_65_val / total_population) * 100

    print(f"Corrected Median Weekly Household Income: ${median_income_val}")
    print(f"Calculated % Over 65: {pct_over_65:.2f}%")
except Exception as e:
    print(f"Error refining demographics: {e}")
    median_income_val = "Error"
    pct_over_65 = None

# 3. Update the features DataFrame
features.loc[0, 'median_income'] = median_income_val
features.loc[0, 'pct_over_65'] = f"{pct_over_65:.2f}%" if pct_over_65 else None
features.loc[0, 'gp_visits_per_capita'] = "MM1 Benchmark (Metro)"

print("\nRefined Feature DataFrame:")
display(features)

Corrected Median Weekly Household Income: $2196.0
Calculated % Over 65: 9.39%

Refined Feature DataFrame:


,postcode,population,median_income,pct_over_65,gp_count_3km,pharmacy_count_3km,gp_visits_per_capita
0,3067,9096,2196.0,9.39%,60,58,MM1 Benchmark (Metro)


### 5.2 Coordinate systems

In [68]:
postcode_geom_vicgrid = postcode_geom.to_crs(epsg=7899)  # metres for area/distance[web:34]
area_m2 = postcode_geom_vicgrid.geometry.area.iloc[0]

### 5.3 Outlier detection on POI counts

In [69]:
poi_counts = pd.DataFrame({
    "gp_count_3km": [len(gp_places)],
    "pharmacy_count_3km": [len(pharmacy_places)],
    # you can add more radii or types across a broader dataset if modelling multiple postcodes
})

iso = IsolationForest(contamination=0.05, random_state=42)
poi_counts["outlier_flag"] = iso.fit_predict(poi_counts)

if poi_counts["outlier_flag"].iloc[0] == -1:
    print("Warning: POI counts look anomalous – check tagging and radius choice.")

## Step 6 – Exploratory data analysis & mapping

### 6.1 Descriptive statistics

In [70]:
print(features.describe(include="all"))

       postcode  population  median_income pct_over_65  gp_count_3km  \
count         1         1.0            1.0           1           1.0   
unique        1         NaN            1.0           1           NaN   
top        3067         NaN         2196.0       9.39%           NaN   
freq          1         NaN            1.0           1           NaN   
mean        NaN      9096.0            NaN         NaN          60.0   
std         NaN         NaN            NaN         NaN           NaN   
min         NaN      9096.0            NaN         NaN          60.0   
25%         NaN      9096.0            NaN         NaN          60.0   
50%         NaN      9096.0            NaN         NaN          60.0   
75%         NaN      9096.0            NaN         NaN          60.0   
max         NaN      9096.0            NaN         NaN          60.0   

        pharmacy_count_3km   gp_visits_per_capita  
count                  1.0                      1  
unique                 NaN     

### 6.2 Folium maps for clinics and pharmacies

In [71]:
m = folium.Map(location=[lat, lon], zoom_start=13)

# Plot GP practices
for place in gp_places:
    p_lat = place["geometry"]["location"]["lat"]
    p_lon = place["geometry"]["location"]["lng"]
    name = place.get("name", "GP practice")
    folium.CircleMarker(
        location=[p_lat, p_lon],
        radius=5,
        color="blue",
        fill=True,
        popup=name,
    ).add_to(m)

# Plot pharmacies
for place in pharmacy_places:
    p_lat = place["geometry"]["location"]["lat"]
    p_lon = place["geometry"]["location"]["lng"]
    name = place.get("name", "Pharmacy")
    folium.CircleMarker(
        location=[p_lat, p_lon],
        radius=5,
        color="green",
        fill=True,
        popup=name,
    ).add_to(m)

# Add postcode boundary
folium.GeoJson(postcode_geom.__geo_interface__, name="postcode").add_to(m)


## Step 7 – Demand modelling (clustering + regression)

### 7.1 Clustering postcode profiles

In [73]:
# To perform clustering, we need a dataset containing multiple postcodes.
# Currently, we only have data for the target postcode (3067).

# For demonstration, we will treat 'features' as our dataset.
# In a full analysis, you would append multiple postcode features to this list.
df_all_postcodes = features.copy()

# Convert percentage string to float for clustering
if isinstance(df_all_postcodes['pct_over_65'].iloc[0], str):
    df_all_postcodes['pct_over_65'] = df_all_postcodes['pct_over_65'].str.rstrip('%').astype('float') / 100.0

# We exclude non-numeric/benchmark columns for the actual algorithm
cluster_cols = ["population", "pct_over_65", "median_income", "gp_count_3km", "pharmacy_count_3km"]

# KMeans requires at least as many rows as clusters.
# Since we only have 1 postcode right now, we can't cluster yet.
if len(df_all_postcodes) > 1:
    kmeans = KMeans(n_clusters=min(len(df_all_postcodes), 4), random_state=42)
    df_all_postcodes["cluster"] = kmeans.fit_predict(df_all_postcodes[cluster_cols])
    print("Clustering completed.")
else:
    print("Note: Clustering skipped. You need more than one postcode in 'df_all_postcodes' to perform comparison.")

display(df_all_postcodes)

Note: Clustering skipped. You need more than one postcode in 'df_all_postcodes' to perform comparison.


,postcode,population,median_income,pct_over_65,gp_count_3km,pharmacy_count_3km,gp_visits_per_capita
0,3067,9096,2196.0,0.0939,60,58,MM1 Benchmark (Metro)


### 7.2 Regression‑style demand score

In [87]:
# Using the scaled features and merged data from the previous clustering step
# We'll use 'gp_count' as a proxy for 'demand' for demonstration purposes since
# specific visit data per postcode isn't available in the current dataframe.

X = scaled_features
y = cluster_df['gp_count']

rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X, y)

# Prepare features for the target site (Abbotsford - index 0)
target_X = scaled_features[0:1]
predicted_score = rf.predict(target_X)[0]

print(f"Modeled supply/demand score for {target_postcode}: {predicted_score}")
print("Note: Regression with 3 data points is for demonstration of the workflow only.")

Modeled supply/demand score for 3067: 60.0
Note: Regression with 3 data points is for demonstration of the workflow only.


### 8. Expanding the Dataset
To improve the reliability of the clustering and regression, we will now add the requested suburbs to our comparative framework.

In [88]:
expanded_postcodes = ["3067", "3066", "3121", "3068", "3070", "3078", "3079", "3101", "3122", "3123"]

# 1. Update multi_poa and locations
multi_poa_ext = poa[poa["POA_CODE21"].isin(expanded_postcodes)].copy().to_crs(epsg=7844)
postcode_locations_ext = {}

for pc in expanded_postcodes:
    geom = multi_poa_ext[multi_poa_ext["POA_CODE21"] == pc]
    if not geom.empty:
        proj_geom = geom.to_crs(epsg=7855)
        c = proj_geom.geometry.centroid.to_crs(epsg=7844).iloc[0]
        postcode_locations_ext[pc] = {"lat": c.y, "lon": c.x}

# 2. Batch fetch supply data
expanded_supply = []
for pc, loc in postcode_locations_ext.items():
    print(f"Fetching data for {pc}...")
    docs, _ = google_nearby_places(loc['lat'], loc['lon'], radius_m=2000, place_type="doctor")
    pharms, _ = google_nearby_places(loc['lat'], loc['lon'], radius_m=2000, place_type="pharmacy")
    expanded_supply.append({
        "postcode": pc,
        "gp_count": len(docs),
        "pharmacy_count": len(pharms)
    })

expanded_supply_df = pd.DataFrame(expanded_supply)

# 3. Batch extract demographics from Census files
expanded_demo = []
target_ids_ext = [f"POA{pc}" for pc in expanded_postcodes]

for poa_id in target_ids_ext:
    row_g01 = df_g01_all[df_g01_all.iloc[:, 0] == poa_id]
    row_g02 = df_g02_all[df_g02_all.iloc[:, 0] == poa_id]

    if not row_g01.empty and not row_g02.empty:
        pop = row_g01['Tot_P_P'].values[0]
        over_65 = row_g01[age_cols].sum(axis=1).values[0]
        med_inc = row_g02['Median_tot_hhd_inc_weekly'].values[0]
        expanded_demo.append({
            "postcode": poa_id.replace("POA", ""),
            "population": pop,
            "median_income": med_inc,
            "pct_over_65": (over_65 / pop) * 100 if pop > 0 else 0
        })

expanded_final_df = pd.merge(expanded_supply_df, pd.DataFrame(expanded_demo), on="postcode")
display(expanded_final_df)

Fetching data for 3067...
Fetching Google Places for doctor...
Fetching Google Places for pharmacy...
Fetching data for 3066...
Fetching Google Places for doctor...
Fetching Google Places for pharmacy...
Fetching data for 3121...
Fetching Google Places for doctor...
Fetching Google Places for pharmacy...
Fetching data for 3068...
Fetching Google Places for doctor...
Fetching Google Places for pharmacy...
Fetching data for 3070...
Fetching Google Places for doctor...
Fetching Google Places for pharmacy...
Fetching data for 3078...
Fetching Google Places for doctor...
Fetching Google Places for pharmacy...
Fetching data for 3079...
Fetching Google Places for doctor...
Fetching Google Places for pharmacy...
Fetching data for 3101...
Fetching Google Places for doctor...
Fetching Google Places for pharmacy...
Fetching data for 3122...
Fetching Google Places for doctor...
Fetching Google Places for pharmacy...
Fetching data for 3123...
Fetching Google Places for doctor...
Fetching Google Pla

,postcode,gp_count,pharmacy_count,population,median_income,pct_over_65
0,3067,60,30,9096,2196,9.388742
1,3066,60,38,9179,2130,7.081381
2,3121,60,24,31534,2283,10.924716
3,3068,60,20,19382,2406,15.297699
4,3070,60,15,25276,2287,13.138946
5,3078,30,7,12237,2166,13.720683
6,3079,60,7,17055,2328,18.334799
7,3101,60,10,24499,2497,19.368137
8,3122,60,13,22322,2145,14.335633
9,3123,60,16,14834,2253,15.147634


### 9. Re-running the Model
Now we re-run the scaling and regression with the expanded dataset.

In [89]:
# Scale new features
final_scaler = StandardScaler()
final_X = final_scaler.fit_transform(expanded_final_df[cluster_cols])
final_y = expanded_final_df['gp_count']

# Re-train Random Forest
final_rf = RandomForestRegressor(n_estimators=200, random_state=42)
final_rf.fit(final_X, final_y)

# Predict for target site (3067 is at index 0)
target_pred = final_rf.predict(final_X[0:1])[0]

print(f"Robust predicted supply/demand score for 3067: {target_pred:.2f}")

# Show Feature Importance
importances = pd.Series(final_rf.feature_importances_, index=cluster_cols)
print("\nModel Logic (Feature Importance):")
print(importances.sort_values(ascending=False))

Robust predicted supply/demand score for 3067: 58.65

Model Logic (Feature Importance):
gp_count          0.818182
pharmacy_count    0.082645
population        0.057851
median_income     0.041322
pct_over_65       0.000000
dtype: float64


In [94]:
# Helper function to fetch various amenities for foot traffic analysis
def get_nearby_amenities(lat, lon, radius=500, types=[]):
    results = {}
    for t in types:
        places, _ = google_nearby_places(lat, lon, radius_m=radius, place_type=t)
        results[t] = places
    return results

# ========== Site Analysis & Scorecard ==========
# 1. Site-specific amenities & brand classification
site_amenities = get_nearby_amenities(s_lat, s_lon, radius=500,
                                      types=['pharmacy','doctor','supermarket','cafe','transit_station','school','gym'])
print("Foot traffic indicators within 500m:")
for k,v in site_amenities.items():
    print(f"  {k}: {len(v)}")

# 2. Pharmacy brand breakdown
def classify_pharmacy(name):
    name = name.lower()
    if 'chemist warehouse' in name: return 'Chemist Warehouse'
    elif 'priceline' in name: return 'Priceline'
    elif 'terry white' in name: return 'Terry White'
    elif 'amcal' in name: return 'Amcal'
    elif 'bloom' in name: return 'Bloom'
    else: return 'Independent'

pharmacy_brands = []
for place in pharmacy_places:
    name = place.get('name', '')
    brand = classify_pharmacy(name)
    pharmacy_brands.append({'name': name, 'brand': brand,
                            'lat': place['geometry']['location']['lat'],
                            'lng': place['geometry']['location']['lng']})
brand_df = pd.DataFrame(pharmacy_brands)
print("\nPharmacy brand distribution in 3km:")
display(brand_df['brand'].value_counts())

# 3. Demand estimation
avg_consults = 5.5
if 'pct_over_65' in features.columns and features['pct_over_65'].iloc[0] is not None:
    pct_val = features['pct_over_65'].iloc[0]
    pct = float(pct_val.rstrip('%')) if isinstance(pct_val, str) else pct_val
    avg_consults += 0.03 * pct

pop = total_population
annual_consult_demand = pop * avg_consults
gp_capacity_per_fte = 5000
existing_gps = len(gp_places)
existing_capacity = existing_gps * gp_capacity_per_fte
gap = annual_consult_demand - existing_capacity
additional_gps_needed = gap / gp_capacity_per_fte

print("\n--- GP Demand & Supply ---")
print(f"Population (postcode): {pop:,}")
print(f"Estimated annual GP consultations needed: {annual_consult_demand:,.0f}")
print(f"Existing GPs in 3km: {existing_gps}")
print(f"Existing annual capacity: {existing_capacity:,.0f}")
print(f"Gap (demand - supply): {gap:,.0f} consultations")
print(f"Additional FTE GPs required: {additional_gps_needed:.1f}")

# 4. Pharmacy scripts
scripts_per_capita = 10
annual_scripts = pop * scripts_per_capita
pharm_count = len(pharmacy_places)
scripts_per_pharm = annual_scripts / pharm_count if pharm_count > 0 else 0
print("\n--- Pharmacy Demand ---")
print(f"Estimated annual prescriptions: {annual_scripts:,.0f}")
print(f"Pharmacies in 3km: {pharm_count}")
print(f"Scripts per pharmacy (if evenly split): {scripts_per_pharm:,.0f}")

# 5. Benchmarking with expanded dataset
if 'final_rf' in locals():
    expanded_final_df['predicted_gp'] = final_rf.predict(final_X)
    expanded_final_df['residual'] = expanded_final_df['gp_count'] - expanded_final_df['predicted_gp']
    target_residual = expanded_final_df[expanded_final_df['postcode']=='3067']['residual'].values[0]
    print(f"\nTarget area residual (actual - predicted GP count): {target_residual:.1f}")

# 6. Go/No-Go scorecard
print("\n===== EXECUTIVE SCORECARD =====")
score = 0
if additional_gps_needed >= 5: score += 2; print("✓ GP demand supports 5 FTE: YES (+2)")
else: score += 1; print("✗ GP demand supports 5 FTE: NO (needs to attract outside) (+1)")

if scripts_per_pharm > 60000: score += 2; print("✓ Pharmacy scripts per pharmacy >60k: YES (+2)")
else: print("✗ Pharmacy scripts per pharmacy >60k: NO (+0)")

if brand_df['brand'].value_counts().get('Chemist Warehouse', 0) < 3: score += 2; print("✓ Fewer than 3 Chemist Warehouses nearby: YES (+2)")
else: print("✗ Chemist Warehouse dominance: NO (+0)")

if len(site_amenities['supermarket']) > 0 and len(site_amenities['cafe']) > 5: score += 2; print("✓ Strong foot traffic indicators: YES (+2)")
else: score += 1; print("✗ Weak foot traffic: NO (+1)")

print(f"\nTotal Score: {score + 1}/10 (Includes parking placeholder)")

Fetching Google Places for pharmacy...
Fetching Google Places for doctor...
Fetching Google Places for supermarket...
Fetching Google Places for cafe...
Fetching Google Places for transit_station...
Fetching Google Places for school...
Fetching Google Places for gym...
Foot traffic indicators within 500m:
  pharmacy: 2
  doctor: 3
  supermarket: 3
  cafe: 12
  transit_station: 24
  school: 2
  gym: 9

Pharmacy brand distribution in 3km:


,count
brand,
Independent,46
Chemist Warehouse,9
Priceline,2
Amcal,1



--- GP Demand & Supply ---
Population (postcode): 9,096
Estimated annual GP consultations needed: 52,590
Existing GPs in 3km: 60
Existing annual capacity: 300,000
Gap (demand - supply): -247,410 consultations
Additional FTE GPs required: -49.5

--- Pharmacy Demand ---
Estimated annual prescriptions: 90,960
Pharmacies in 3km: 58
Scripts per pharmacy (if evenly split): 1,568

Target area residual (actual - predicted GP count): 1.4

===== EXECUTIVE SCORECARD =====
✗ GP demand supports 5 FTE: NO (needs to attract outside) (+1)
✗ Pharmacy scripts per pharmacy >60k: NO (+0)
✗ Chemist Warehouse dominance: NO (+0)
✓ Strong foot traffic indicators: YES (+2)

Total Score: 4/10 (Includes parking placeholder)


In [95]:
# Use catchment_pop instead of total_population
annual_consult_demand = catchment_pop * avg_consults   # avg_consults as before (5.5 + age adjustment)
existing_gps = len(gp_places)  # still from the 3km search – but now we have the correct population
existing_capacity = existing_gps * gp_capacity_per_fte
gap = annual_consult_demand - existing_capacity
additional_gps_needed = gap / gp_capacity_per_fte

print(f"Catchment population: {catchment_pop:,}")
print(f"Annual demand: {annual_consult_demand:,.0f}")
print(f"Existing GP capacity: {existing_capacity:,.0f}")
print(f"Gap: {gap:,.0f}")
print(f"Additional FTE GPs required: {additional_gps_needed:.1f}")

# Pharmacy scripts
scripts_per_capita = 10  # still a reasonable metro average
annual_scripts = catchment_pop * scripts_per_capita
scripts_per_pharm = annual_scripts / len(pharmacy_places) if len(pharmacy_places) > 0 else 0
print(f"Annual scripts: {annual_scripts:,.0f}")
print(f"Scripts per pharmacy: {scripts_per_pharm:,.0f}")

Catchment population: 153,525
Annual demand: 887,635
Existing GP capacity: 300,000
Gap: 587,635
Additional FTE GPs required: 117.5
Annual scripts: 1,535,250
Scripts per pharmacy: 26,470


In [97]:
# Convert GP places to GeoDataFrame
gp_locations = []
for place in gp_places:
    lat = place['geometry']['location']['lat']
    lng = place['geometry']['location']['lng']
    gp_locations.append({'lat': lat, 'lng': lng})

# Fix: Create a DataFrame from the list first
gp_loc_df = pd.DataFrame(gp_locations)
gps_gdf = gpd.GeoDataFrame(gp_loc_df, geometry=gpd.points_from_xy(gp_loc_df['lng'], gp_loc_df['lat']), crs="EPSG:4326")
gps_gdf = gps_gdf.to_crs(epsg=7844)

# Spatial join with postcode polygons (using all postcodes multi_poa_ext)
gps_with_poa = gpd.sjoin(gps_gdf, multi_poa_ext, how='inner', predicate='within')
gp_counts = gps_with_poa.groupby('POA_CODE21').size().reset_index(name='gp_count_postcode')

# Process pharmacies
pharm_locations = []
for place in pharmacy_places:
    lat = place['geometry']['location']['lat']
    lng = place['geometry']['location']['lng']
    pharm_locations.append({'lat': lat, 'lng': lng})

pharm_loc_df = pd.DataFrame(pharm_locations)
pharms_gdf = gpd.GeoDataFrame(pharm_loc_df, geometry=gpd.points_from_xy(pharm_loc_df['lng'], pharm_loc_df['lat']), crs="EPSG:4326")
pharms_gdf = pharms_gdf.to_crs(epsg=7844)

pharms_with_poa = gpd.sjoin(pharms_gdf, multi_poa_ext, how='inner', predicate='within')
pharm_counts = pharms_with_poa.groupby('POA_CODE21').size().reset_index(name='pharmacy_count_postcode')

# Merge results
expanded_final_df = expanded_final_df.merge(gp_counts, left_on='postcode', right_on='POA_CODE21', how='left')
expanded_final_df = expanded_final_df.merge(pharm_counts, left_on='postcode', right_on='POA_CODE21', how='left')

expanded_final_df['gp_count_postcode'] = expanded_final_df['gp_count_postcode'].fillna(0)
expanded_final_df['pharmacy_count_postcode'] = expanded_final_df['pharmacy_count_postcode'].fillna(0)

display(expanded_final_df[['postcode', 'gp_count_postcode', 'pharmacy_count_postcode']].head())

,postcode,gp_count_postcode,pharmacy_count_postcode
0,3067,0.0,2.0
1,3066,2.0,5.0
2,3121,27.0,13.0
3,3068,2.0,3.0
4,3070,0.0,1.0
